In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
##Read and load the dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()

In [ ]:
## Preprocessing the data
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

In [ ]:
##Encoding categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

In [ ]:
#OHE for Geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


In [ ]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

In [ ]:
pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [ ]:
##combine OHE columns with original dataset
data = pd.concat(
    [data.drop('Geography', axis=1),
     pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))],
    axis=1
)
data.head()

In [ ]:
## Save the encoders and scaler for future use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [ ]:
## Divide the dataset into dependent and independent features
X = data.drop('Exited', axis=1)
y = data['Exited']

## Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# get numpy array from the DataFrame
X_train_array = X_train.to_numpy()
X_train_array

In [ ]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

ANN Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [ ]:
## Building the ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)),## HL 1 Connected to input layer
    Dense(32, activation='relu'),## HL 2
    Dense(1, activation='sigmoid')## Output layer
])

In [ ]:
model.summary()

In [ ]:
## compiling the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
## Setting up TensorBoard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [ ]:
##Setting up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
## Training the model
history = model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=100,callbacks=[early_stopping_callback, tensorflow_callback])

In [ ]:
model.save('model.h5')

In [ ]:
## Load TensorBoard Extension
%load_ext tensorboard

In [ ]:
%pip install setuptools


In [ ]:
%tensorboard --logdir logs/fit